KPIs.
### 1. TOTAL SALES:
Total money collected from customers for all products sold.

### 2. PROFIT PER PRODUCT:
Selling price minus the cost of the item.

### 3. TOP SELLING CATEGORY:
Which group (e.g. Electronics) brings the most money.

### 4. BASKET SIZE:
Average number of items bought per single transaction.

### 5. RETURN RATE:
Percentage of total sales that were returned by customers.

### 6. OUT-OF-STOCK COUNT:
Count of unique products with zero units in stock.

### 7. STORE PERFORMANCE:
Ranking of the 5 stores based on daily sales volume.

### 8. DISCOUNT IMPACT:
Money lost due to differences in marked vs. sold price.

### 9. REPEAT CUSTOMER COUNT:
Number of unique users with more than one purchase.

### 10. SLOW-MOVING INVENTORY:
Products with zero sales recorded in the last 30 days.

In [0]:
USE workspace.gold;

-- KPI 1
CREATE OR REPLACE VIEW views.total_sales AS (
  SELECT sum(total_amount) as total_sales FROM fact_sales
)

In [0]:
--- KPI 2
-- Profit Per Product
CREATE OR REPLACE VIEW views.profit_per_product AS
with raw_profit_per_product as (
  SELECT
    product_id,
    SUM(profit_amount) as total_profit
  FROM
    fact_sales
  GROUP BY
    product_id
),
profit_per_product as (
  select
    products.product_id,
    products.item_name_description as product_name,
    ifnull(ppp.total_profit, 0)as total_profit
  from
    dim_products as products left join raw_profit_per_product as ppp using (product_id)
)
SELECT
  *
FROM
  profit_per_product;

In [0]:
-- KPI 3
CREATE OR REPLACE VIEW views.profit_per_category AS
(
  select
    category_type,
    sum(total_profit) as total_profit,
    row_number() over (order by sum(total_profit) desc) as rank
  from
    views.profit_per_product left join dim_products using (product_id)
  group by
    dim_products.category_type
  order by
    total_profit desc
);

In [0]:
-- KPI 4
-- Bucket Size
CREATE OR REPLACE VIEW views.average_bucket_size AS
select
  AVG(qty_sold) as bucket_size
from
  fact_sales;

In [0]:
-- KPI 5
-- Return Rate
CREATE OR REPLACE VIEW views.return_rate AS
select
  round((count(*) - (select count(*) from fact_returns)) / count(*), 2) as return_rate
from
  fact_sales;

In [0]:
-- KPI 6
-- Out of stock
create or replace view views.out_of_stock as
(
  select
    product_id,
    store_id,
    dim_stores.location_name_address as store_name,
    dim_products.item_name_description as product_name
  from
    fact_inv_levels
    left join dim_products using (product_id)
    left join dim_stores using (store_id)
  where
    stock_on_hand = 0
  group by
    product_id,
    store_id,
    dim_stores.location_name_address,
    dim_products.item_name_description
);

In [0]:
-- KPI 7
-- STORE PERFORMANCE
CREATE OR REPLACE VIEW views.store_performance as
(
  select
    store_id,
    dim_stores.location_name_address as store_name,
    round(avg(qty_sold), 2) as daily_volume,
    row_number() over (order by avg(qty_sold) desc) as rank
  from
    fact_sales
    left join dim_stores using (store_id)
  group by
    store_id, store_name
  order by
    daily_volume desc
)

In [0]:
-- KPI 8
---  DISCOUNT IMPACT
create or replace view views.total_discount_impact as
(
  select
    sum(discounted_amount) as total_discount
  from
    fact_sales
);

create or replace view views.total_discount_per_store as
(
  select
    store_id,
    dim_stores.location_name_address as store_name,
    sum(discounted_amount)
  from
    fact_sales
    left join dim_stores using (store_id)
  group by
    store_id, dim_stores.location_name_address
)


In [0]:
-- KPI 9
-- Returning Customers
create or replace view views.total_returning_customers as
(
  select
    count(distinct cust_ref_id) as returning_customers
  from
    fact_sales
);

CREATE OR REPLACE VIEW views.returning_customer_per_store AS
SELECT 
    store_id,
    store_name,
    COUNT(cust_ref_id) AS returning_customers
FROM (
    SELECT 
        s.store_id, 
        ds.location_name_address AS store_name, 
        s.cust_ref_id
    FROM fact_sales s
    JOIN dim_stores ds ON s.store_id = ds.store_id
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
) 
GROUP BY 1, 2;

In [0]:
-- kpi 10
-- slow moving inventory
create or replace view views.slow_moving_inventory as
with products_last_30_days as (
  select
    product_id,
    sale_date
  from
    fact_sales
  where
    date_diff(day, sale_date, current_date()) < 60
  order by
    sale_date desc
),
sold_products as (
  select distinct
    product_id,
    sale_date
  from
    products_last_30_days
),
slow_moving_inventory as (
  select
    product_id,
    dim_products.item_name_description as product_name,
    dim_products.category_type as category
  from
    fact_inv_levels as inv
    left join sold_products using (product_id)
    left join dim_products using (product_id)
  where
    inv.product_id not in (select product_id from sold_products)
)
select
  *
from
  slow_moving_inventory;